# 01 — Synthetic Data Generator

This notebook creates literature-grounded synthetic menstrual cycle data for the Girls & Cycles project.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [ ]:
N_USERS = 500
CYCLES_PER_USER = 12
N = N_USERS * CYCLES_PER_USER

# BMI categories and irregularity rates (Patel 2023 / Fitriani 2019)
bmi_categories = np.random.choice(["underweight", "normal", "overweight"], size=N, p=[0.15, 0.60, 0.25])
irregularity_rate = {"underweight": 0.318, "normal": 0.184, "overweight": 0.266}
dysmenorrhea_rate = {"underweight": 0.903, "normal": 0.307, "overweight": 0.45}

is_irregular = np.array([np.random.rand() < irregularity_rate[c] for c in bmi_categories])
has_dysmenorrhea = np.array([np.random.rand() < dysmenorrhea_rate[c] for c in bmi_categories])

In [ ]:
# Baseline cycle length grounding (Bull et al. 2019)
base_cycle_length = np.random.normal(loc=30.17, scale=6.88, size=N)

# Irregular users have increased variance/noise
noise = np.where(is_irregular, np.random.normal(0, 3.5, N), np.random.normal(0, 1.5, N))
cycle_length_days = np.clip(base_cycle_length + noise, 20, 45)

stress_score = np.clip(np.random.normal(5.5, 2.0, N) + is_irregular * 1.0, 0, 10)
sleep_quality = np.clip(np.random.normal(6.5, 1.8, N) - is_irregular * 0.7, 1, 10)

In [ ]:
df = pd.DataFrame({
    "user_id": np.repeat(np.arange(1, N_USERS + 1), CYCLES_PER_USER),
    "cycle_index": np.tile(np.arange(1, CYCLES_PER_USER + 1), N_USERS),
    "bmi_category": bmi_categories,
    "is_irregular": is_irregular.astype(int),
    "has_dysmenorrhea": has_dysmenorrhea.astype(int),
    "stress_score": np.round(stress_score, 2),
    "sleep_quality": np.round(sleep_quality, 2),
    "cycle_length_days": np.round(cycle_length_days, 2)
})

df.head()

In [ ]:
summary = df.groupby("bmi_category")["is_irregular"].mean().mul(100).round(1)
print("Irregularity rate by BMI category (%):")
print(summary)

print("\nSynthetic dataset shape:", df.shape)

In [ ]:
# Save locally if desired (optional)
# df.to_csv("../data/synthetic_cycles.csv", index=False)